# Imports

In [1]:
#Base imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Preprocessing imports
from imputers import MasterImputer
from sklearn.preprocessing import StandardScaler

# Modeling imports
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression



# Exploratory Data Analysis

In [2]:
data = pd.read_csv('ctd-datawiz-2023/train.csv')
X = data.drop(columns=['Attrition'])
y = data['Attrition']


In [3]:
X.describe()

,User ID,Mobile No.,Duration,Voicemail_Messages,Day_Minutes,Day_Calls,Day_Charge,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls
count,4150.000000,4.150000e+03,4150.000000,3681.000000,4150.000000,3640.000000,4150.000000,3626.000000,4150.000000,4150.000000,3548.000000,4150.000000,4150.000000,4150.000000,4150.000000,3561.000000,4150.000000
mean,2776.621928,7.988843e+09,99.967470,7.890519,177.145687,100.010440,391.499386,199.429923,100.250120,220.259219,199.746167,99.983855,116.795822,10.206482,4.455181,35.851342,1.495904
std,1346.549853,1.157382e+09,39.816089,13.599601,51.396781,19.569463,113.586282,50.015577,19.845768,55.418443,50.492036,19.898864,29.707936,2.759975,2.436228,9.615616,1.211282
min,406.000000,6.000109e+09,1.000000,0.000000,0.000000,0.000000,0.000000,22.300000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1589.250000,6.985293e+09,73.000000,0.000000,142.825000,87.000000,315.672500,165.125000,87.000000,182.292500,165.975000,87.000000,97.110000,8.500000,3.000000,29.900000,1.000000
50%,2830.500000,7.958110e+09,100.000000,0.000000,178.400000,100.000000,394.290000,199.800000,101.000000,220.805000,199.150000,100.000000,116.740000,10.300000,4.000000,36.140000,1.000000
75%,3962.750000,9.006551e+09,127.000000,18.000000,212.100000,113.000000,468.780000,232.900000,114.000000,257.400000,233.325000,113.000000,136.467500,12.000000,6.000000,42.120000,2.000000
max,5000.000000,9.998408e+09,243.000000,52.000000,351.500000,163.000000,776.880000,361.800000,170.000000,399.750000,395.000000,175.000000,231.010000,19.700000,19.000000,69.160000,8.000000


In [4]:
X.isna().sum()

User ID                 0
Mobile No.              0
User DOB                0
User Job                0
Duration                0
Intl_Plan             471
Voicemail_Plan          0
Voicemail_Messages    469
Day_Minutes             0
Day_Calls             510
Day_Charge              0
Eve_Minutes           524
Eve_Calls               0
Eve_Charge              0
Night_Minutes         602
Night_Calls             0
Night_Charge            0
Intl_Minutes            0
Intl_Calls              0
Intl_Charge           589
Service_Calls           0
dtype: int64

Areas of Note:
- 469 people got N/A as voicemail messages. This needs to be cleaned to most likely say 0 because these people did not have a voicemail plan. These 2 variables are likely to be highly correlated.
- 471 people have N/A on if they have an international plan (yes/no)
- 510 people have N/A as daycalls
- 524 people have N/A as eve_minutes
- 602 people have N/A as night_minutes
- 589 peopl have N/A for international charges


Somethings to clean data 

- if N/A for international charges and have no international plan -> 0, otherwise 1

- if N/A international plan and have international charges then plan ->yes otherwise no

- for the day calls find median time for day_minutes and divide the minites spent by that to find number of calls
    for eve_minutes and night_minutes do the opposite way

- if has voicemail plan -> median number of voicemessages, otherwise 0.




## Data Imputation

In [5]:
imputer = MasterImputer(voicemail_strategy='zero', minute_strategy='mean', charge_strategy='mean')
X = imputer.fit_transform(X)

In [6]:
X.isna().sum()

User ID               0
Mobile No.            0
User DOB              0
User Job              0
Duration              0
Intl_Plan             0
Voicemail_Plan        0
Voicemail_Messages    0
Day_Minutes           0
Day_Calls             0
Day_Charge            0
Eve_Minutes           0
Eve_Calls             0
Eve_Charge            0
Night_Minutes         0
Night_Calls           0
Night_Charge          0
Intl_Minutes          0
Intl_Calls            0
Intl_Charge           0
Service_Calls         0
dtype: int64

## Feature Selection

In [7]:
(X[X["User Job"].duplicated(keep=False)]).describe()

,User ID,Mobile No.,Duration,Voicemail_Messages,Day_Minutes,Day_Calls,Day_Charge,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls
count,4140.000000,4.140000e+03,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000
mean,2775.965217,7.988147e+09,99.943478,6.985507,177.130242,100.068841,391.465232,199.320146,100.247101,220.251527,199.656020,99.977295,116.800164,10.201473,4.455797,35.814129,1.495169
std,1346.870957,1.157634e+09,39.835221,13.047585,51.412957,18.729106,113.622049,50.142101,19.853560,55.407051,50.790006,19.912953,29.712423,2.759667,2.436679,9.685785,1.211097
min,406.000000,6.000109e+09,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1587.750000,6.983676e+09,73.000000,0.000000,142.800000,88.000000,315.640000,164.984702,87.000000,182.357500,165.973433,87.000000,97.110000,8.500000,3.000000,29.900000,1.000000
50%,2828.500000,7.955418e+09,100.000000,0.000000,178.400000,100.000000,394.290000,199.800000,101.000000,220.740000,199.500000,100.000000,116.740000,10.300000,4.000000,36.140000,1.000000
75%,3962.250000,9.007061e+09,127.000000,0.000000,212.100000,112.000000,468.780000,232.825000,114.000000,257.302500,233.307599,113.000000,136.500000,12.000000,6.000000,42.120000,2.000000
max,5000.000000,9.998408e+09,243.000000,52.000000,351.500000,163.000000,776.880000,361.800000,170.000000,399.750000,395.000000,175.000000,231.010000,19.700000,19.000000,69.162475,8.000000


Only 10 people have the same job description. 
<br>At a later stage for potential improved performance we can change the users to fields from there position. for example engineer, manager etc.

In [8]:
X.head().describe()

,User ID,Mobile No.,Duration,Voicemail_Messages,Day_Minutes,Day_Calls,Day_Charge,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls
count,5.000000,5.000000e+00,5.000000,5.0,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,3499.000000,8.266415e+09,97.600000,0.0,204.520000,92.400000,451.984000,210.434825,102.800000,232.518000,222.700000,115.000000,130.286000,11.580000,3.800000,40.639438,2.600000
std,15.890249,1.232439e+09,46.003261,0.0,78.907237,8.961027,174.390758,39.220457,21.393924,43.335515,50.163682,24.889757,29.365386,1.333042,2.683282,4.683687,1.516575
min,3487.000000,6.537028e+09,40.000000,0.0,115.600000,81.000000,255.450000,163.174127,78.000000,180.310000,151.400000,83.000000,88.530000,9.700000,1.000000,34.060000,1.000000
25%,3488.000000,7.741629e+09,69.000000,0.0,159.600000,86.000000,352.690000,196.000000,89.000000,216.580000,197.100000,104.000000,115.310000,11.200000,1.000000,39.260000,2.000000
50%,3495.000000,8.157186e+09,104.000000,0.0,176.800000,93.000000,390.780000,209.100000,98.000000,231.010000,232.600000,116.000000,136.110000,11.300000,5.000000,39.650000,2.000000
75%,3499.000000,9.399738e+09,114.000000,0.0,261.200000,100.000000,577.200000,212.600000,119.000000,234.910000,251.300000,121.000000,147.030000,12.600000,5.000000,44.235898,3.000000
max,3526.000000,9.496494e+09,161.000000,0.0,309.400000,102.000000,683.800000,271.300000,130.000000,299.780000,281.100000,151.000000,164.450000,13.100000,7.000000,45.991290,5.000000


Notes:
- **User ID** should be dropped for the model
- **User DOB** doesnt matter as much as birth year. The year is going to keep getting newer for new future users. Preventing model degredation wont really matter for this specific variable because the model will be rerun more than atleast once a year
- **Mobile No.** is not importanat but the area code is

- **Duration** can be dropped since its just the combination of all the _Minutes or atleast should be

- Everything else should just be scaled

 

In [9]:
X = X.drop('User ID', axis = 1)
X = X.drop(['User Job','Duration'], axis = 1)

X['Mobile No.'] = X['Mobile No.'].astype(str).str[:3]
X['User DOB'] = X['User DOB'].astype(str).str[:4]

X.rename(columns={
                'Mobile No.': 'Area_Code',
                'User DOB': 'Birth_Year'
                }, inplace=True)

X["Voicemail_Plan"] = X["Voicemail_Plan"] == "Yes"
X["Intl_Plan"] = X["Intl_Plan"] == "Yes"

In [10]:
X.head()

,Area_Code,Birth_Year,Intl_Plan,Voicemail_Plan,Voicemail_Messages,Day_Minutes,Day_Calls,Day_Charge,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls
0,653,1969,False,False,0.0,309.4,86.0,683.80,271.300000,98,299.78,197.1,116,115.31,11.2,5,39.260000,3
1,939,1981,False,False,0.0,159.6,102.0,352.69,212.600000,119,234.91,151.4,83,88.53,12.6,7,44.235898,5
2,949,2002,False,False,0.0,176.8,100.0,390.78,196.000000,89,216.58,232.6,151,136.11,13.1,5,45.991290,1
3,774,1976,False,False,0.0,115.6,93.0,255.45,163.174127,78,180.31,251.3,104,147.03,9.7,1,34.060000,2
4,815,1957,False,False,0.0,261.2,81.0,577.20,209.100000,130,231.01,281.1,121,164.45,11.3,1,39.650000,2


In [11]:
X.Area_Code.unique()

<StringArray>
['653', '939', '949', '774', '815', '946', '671', '936', '834', '644',
 ...
 '845', '750', '756', '935', '689', '639', '948', '947', '824', '611']
Length: 400, dtype: str

Theres still 400 area codes. Since people can keep their phone numbers even after leaving a state it is only a proxy for location. For these reasons it might not be a good predictor. Additionally treating it as a categorical variable could lead to overfitting since it needs to be encodied using 400 seperate variables. For this reason the initial model will be trained without it.

In [12]:
X_no_area = X.drop('Area_Code', axis=1)


In [13]:
X_no_area.head()

,Birth_Year,Intl_Plan,Voicemail_Plan,Voicemail_Messages,Day_Minutes,Day_Calls,Day_Charge,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls
0,1969,False,False,0.0,309.4,86.0,683.80,271.300000,98,299.78,197.1,116,115.31,11.2,5,39.260000,3
1,1981,False,False,0.0,159.6,102.0,352.69,212.600000,119,234.91,151.4,83,88.53,12.6,7,44.235898,5
2,2002,False,False,0.0,176.8,100.0,390.78,196.000000,89,216.58,232.6,151,136.11,13.1,5,45.991290,1
3,1976,False,False,0.0,115.6,93.0,255.45,163.174127,78,180.31,251.3,104,147.03,9.7,1,34.060000,2
4,1957,False,False,0.0,261.2,81.0,577.20,209.100000,130,231.01,281.1,121,164.45,11.3,1,39.650000,2


## Feature Scaling

In [14]:
X_no_area.head()

,Birth_Year,Intl_Plan,Voicemail_Plan,Voicemail_Messages,Day_Minutes,Day_Calls,Day_Charge,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls
0,1969,False,False,0.0,309.4,86.0,683.80,271.300000,98,299.78,197.1,116,115.31,11.2,5,39.260000,3
1,1981,False,False,0.0,159.6,102.0,352.69,212.600000,119,234.91,151.4,83,88.53,12.6,7,44.235898,5
2,2002,False,False,0.0,176.8,100.0,390.78,196.000000,89,216.58,232.6,151,136.11,13.1,5,45.991290,1
3,1976,False,False,0.0,115.6,93.0,255.45,163.174127,78,180.31,251.3,104,147.03,9.7,1,34.060000,2
4,1957,False,False,0.0,261.2,81.0,577.20,209.100000,130,231.01,281.1,121,164.45,11.3,1,39.650000,2


In [15]:
numerical_features = X_no_area.columns.drop(['Intl_Plan', 'Voicemail_Plan'])

std_scaler = StandardScaler()

X_no_area_scaled = pd.DataFrame(
    std_scaler.fit_transform(X_no_area[numerical_features]),
    columns=numerical_features,
    index=X_no_area.index
).join(X_no_area[['Intl_Plan', 'Voicemail_Plan']])


In [16]:
X_no_area_scaled

,Birth_Year,Voicemail_Messages,Day_Minutes,Day_Calls,Day_Charge,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls,Intl_Plan,Voicemail_Plan
0,-0.247103,-0.536398,2.573512,-0.752515,2.573690,1.435257,-0.113394,1.435088,-0.050191,0.804974,-0.050020,0.360017,0.223659,0.353957,1.241889,False,False
1,0.400669,-0.536398,-0.341418,0.101057,-0.341714,0.264684,0.944894,0.264398,-0.950217,-0.853612,-0.951572,0.867329,1.044700,0.867695,2.893231,False,False
2,1.534270,-0.536398,-0.006727,-0.005639,-0.006334,-0.066348,-0.566946,-0.066398,0.648955,2.564081,0.650214,1.048512,0.223659,1.048931,-0.409453,False,False
3,0.130764,-0.536398,-1.197606,-0.379077,-1.197907,-0.720949,-1.121287,-0.720952,1.017237,0.201852,1.017837,-0.183532,-1.418421,-0.182918,0.416218,False,False
4,-0.894875,-0.536398,1.635597,-1.019256,1.635083,0.194888,1.499235,0.194016,1.604125,1.056275,1.604282,0.396253,-1.418421,0.394223,0.416218,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4145,-0.193122,-0.536398,-0.125425,0.047709,-0.125377,-0.124178,1.297656,-0.125050,-1.389398,0.201852,-1.389218,-1.198156,1.044700,-1.198395,-0.409453,False,False
4146,0.022802,2.529262,1.139398,1.434764,1.139453,0.473186,1.297656,0.473199,1.927110,0.804974,1.928141,-0.111059,0.223659,-0.110979,0.416218,False,False
4147,1.102422,-0.536398,-0.711136,-0.592470,-0.711434,-0.528993,1.398445,-0.528574,0.251131,-0.149969,0.251956,1.229695,-0.186861,1.226379,-0.409453,False,False
4148,1.696213,-0.536398,0.226779,-1.766132,0.227173,-0.550929,-0.415762,-0.552035,0.487462,-0.552051,0.488285,-0.618371,0.634179,-0.612418,-1.235124,False,False


In [17]:
y = y=='Yes'

# FULL PREPROCESSING PIPELINE

In [18]:
def preprocessing_pipeline(X): 
    ### Data Imputation
    imputer = MasterImputer(voicemail_strategy='zero', minute_strategy='mean', charge_strategy='mean')

    X = imputer.fit_transform(X)

    ### Data Cleaning
    X = X.drop('User ID', axis = 1)
    X = X.drop(['User Job','Duration'], axis = 1)

    X['Mobile No.'] = X['Mobile No.'].astype(str).str[:3]
    X['User DOB'] = X['User DOB'].astype(str).str[:4]

    X.rename(columns={
                    'Mobile No.': 'Area_Code',
                    'User DOB': 'Birth_Year'
                    }, inplace=True)

    X["Voicemail_Plan"] = X["Voicemail_Plan"] == "Yes"
    X["Intl_Plan"] = X["Intl_Plan"] == "Yes"
    X = X.drop('Area_Code', axis=1)

    ### Data Scaling
    numerical_features = X.columns.drop(['Intl_Plan', 'Voicemail_Plan'])

    std_scaler = StandardScaler()

    X_scaled = pd.DataFrame(
        std_scaler.fit_transform(X[numerical_features]),
        columns=numerical_features,
        index=X.index
    ).join(X[['Intl_Plan', 'Voicemail_Plan']])

    return X_scaled


# Model Selection and Evaluation

In [23]:
# Binary Classifier
data = pd.read_csv('ctd-datawiz-2023/train.csv')
X = data.drop(columns=['Attrition'])
y = data['Attrition']

X_scaled = preprocessing_pipeline(X)

sgd_clf = SGDClassifier(random_state=42)
score = cross_val_score(sgd_clf, X_scaled, y, cv=3)
float(score.mean())

0.9493974103377512

In [29]:
sum(y=='Yes')

207

In [24]:
# Logistic Regression
log_clf = LogisticRegression(max_iter=1000, random_state=42)
score = cross_val_score(log_clf, X_no_area_scaled, y, cv=3)
float(score.mean())


0.9513254116529227

In [ ]:
#This seems to be better performance than the stuff on kaggle.

In [25]:
#Best Model: Logistic Regression
log_clf.fit(X_no_area_scaled, y)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

# Test data 

In [26]:
X_test = pd.read_csv('ctd-datawiz-2023/test.csv')
X_test = preprocessing_pipeline(X_test)
y_pred = log_clf.predict(X_test)

In [28]:
y_pred

array(['No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'N

# clearly we forgot to stratify the train/test split, so we have a class imbalance problem. We need to fix this in the next iteration of the model.